In [243]:
import json
import pandas as pd
import os
from pathlib import Path
from cc import load_coding_df, merge_left, multilabel_metrics

# Metrics — How They Work

This notebook measures how well three coding sources agree with one another.

---

## Three sources compared

| Source | Description |
|---|---|
| **Full-text** (`c_reference_full`) | Gold-standard labels coded from the full paper |
| **Manual abstract** (`c_reference_abs`) | Human-coded labels from abstract only (~100 records) |
| **LLM** (`c_reference_llm`) | Model predictions on the same abstracts |

Each **Consistency Check** is a directed pairwise comparison:

| Check | Left (estimate) | Right (truth) | Records |
|---|---|---|---|
| **CC1** | Manual abstract | Full-text | ~100 |
| **CC2** | LLM | Full-text | all LLM-coded |
| **CC3** | LLM | Manual abstract | ~100 overlap |

The join is always **left JOIN right on record ID**. The left side drives which records are evaluated; unmatched left rows are dropped.

---

## What is included / excluded

**Excluded from all metrics — counted separately in summary:**

| Exclusion | When it happens | Summary row |
|---|---|---|
| `Records excluded (true)` | True label is empty or only `"not applicable"` / `"unclear"` after stripping | Always shown |
| `Records excluded (pred)` | Pred label is empty/special on a record that *has* a valid true label | Always shown |

`Abstention rate` = Records excluded (pred) / (Records matched − Records excluded (true))

`Records evaluated` = Records matched − Records excluded (true) − Records excluded (pred)

**Excluded from aggregate summary metrics only (still visible in per-label table):**

- Labels where **`Support (true) = 0`** — the label never appears in the true set but the estimate predicted it. These "phantom" predictions would distort macro averages if included.

---

## Metrics in the summary

| Metric | What it measures |
|---|---|
| **Exact Match Rate** | Fraction of records where pred frozenset == true frozenset exactly |
| **Jaccard Similarity (macro/micro/weighted)** | Set overlap: `\|true ∩ pred\| / \|true ∪ pred\|` averaged across labels |
| **Kappa (macro/micro/weighted)** | Cohen's kappa per label, then averaged — accounts for chance agreement |
| **Precision / Recall / F1** | Standard multi-label P/R/F1 with macro, micro, and support-weighted averaging |

All aggregate metrics are computed **only over labels with `Support (true) > 0`**.

In [244]:
#  Run name (controls output directory)
RUN_NAME = "180426_gpt5nanoHigh_test"  # set before running
MODE     = "test"      # "train" | "dev" | "test"

run_name_l1= "l1_test_170426_f3"
run_name_l2= "l2_test_170426_f3"
run_name_l3= "l3_test_170426_f3"
run_name_l4= "l4_test_170426_f3"
run_name_l5= "l5_test_170426_f3"
run_name_l6= "l6_test_170426_f3"

In [245]:
# General Paths - do not modify 
C_ANNOTATED_PATH     = "../data/consistency-check-datasets/data-coding/annotated_coding_dataset.xlsx"

C_REFERENCE_FULL_L1      = f"../data/labels/l1/{MODE}/L1) driver set.csv"
C_REFERENCE_FULL_L2      = f"../data/labels/l2/{MODE}/L2) threats set.csv"
C_REFERENCE_FULL_L3      = f"../data/labels/l3/{MODE}/L3) geography set.csv"
# C_REFERENCE_FULL_L3_     = "../data/labels/l3-archive/train/L3) geography set.csv"
C_REFERENCE_FULL_L4      = f"../data/labels/l4/{MODE}/L4) ecosystem set.csv"
C_REFERENCE_FULL_L5      = f"../data/labels/l5/{MODE}/L5) study set.csv"
C_REFERENCE_FULL_L6      = f"../data/labels/l6/{MODE}/L6) taxa set.csv"

C_REFERENCE_ABS_SHEET_L1 = "l1_manual_sample"
C_REFERENCE_ABS_SHEET_L2 = "l2_manual_sample"
C_REFERENCE_ABS_SHEET_L3 = "l3_manual_sample"
C_REFERENCE_ABS_SHEET_L4 = "l4_manual_sample"
C_REFERENCE_ABS_SHEET_L5 = "l5_manual_sample"
C_REFERENCE_ABS_SHEET_L6 = "l6_manual_sample"

# Paths to llm outputs for evaluation
C_REFERENCE_LLM_L1       = f"../data/labels/eval/{run_name_l1}/data/driver.xlsx"
C_REFERENCE_LLM_L2       = f"../data/labels/eval/{run_name_l2}/data/threats_l0.xlsx"
C_REFERENCE_LLM_L3       = f"../data/labels/eval/{run_name_l3}/data/geography.xlsx"
C_REFERENCE_LLM_L4       = f"../data/labels/eval/{run_name_l4}/data/ecosystems_realm.xlsx"
C_REFERENCE_LLM_L5       = f"../data/labels/eval/{run_name_l5}/data/study.xlsx"
C_REFERENCE_LLM_L6       = f"../data/labels/eval/{run_name_l6}/data/taxa.xlsx"

#  Results collection
_results = {
    "L1 — Drivers":      [],
    "L2 — threats_l0":   [],
    "L3 — region":       [],
    "L3 — sub-region":   [],
    "L3 — country":      [],
    "L4 — realm":        [],
    "L5 — study_design": [],
    "L6 — taxa":         [],
    "L6 — phylum":       [],
    "L6 — class":        [],
    "L6 — order":        [],
    "L6 — genus":        [],
    "L6 — specie":       [],
}

#  Per-check path metadata — used by the export cell for traceability 
_check_sources = {}
for _sheet, _ft, _sh, _llm in [
    ("L1 — Drivers",      C_REFERENCE_FULL_L1, C_REFERENCE_ABS_SHEET_L1, C_REFERENCE_LLM_L1),
    ("L2 — threats_l0",   C_REFERENCE_FULL_L2, C_REFERENCE_ABS_SHEET_L2, C_REFERENCE_LLM_L2),
    ("L3 — region",       C_REFERENCE_FULL_L3, C_REFERENCE_ABS_SHEET_L3, C_REFERENCE_LLM_L3),
    ("L3 — sub-region",   C_REFERENCE_FULL_L3, C_REFERENCE_ABS_SHEET_L3, C_REFERENCE_LLM_L3),
    ("L3 — country",      C_REFERENCE_FULL_L3, C_REFERENCE_ABS_SHEET_L3, C_REFERENCE_LLM_L3),
    ("L4 — realm",        C_REFERENCE_FULL_L4, C_REFERENCE_ABS_SHEET_L4, C_REFERENCE_LLM_L4),
    ("L5 — study_design", C_REFERENCE_FULL_L5, C_REFERENCE_ABS_SHEET_L5, C_REFERENCE_LLM_L5),
    ("L6 — taxa",         C_REFERENCE_FULL_L6, C_REFERENCE_ABS_SHEET_L6, C_REFERENCE_LLM_L6),
    ("L6 — phylum",       C_REFERENCE_FULL_L6, C_REFERENCE_ABS_SHEET_L6, C_REFERENCE_LLM_L6),
    ("L6 — class",        C_REFERENCE_FULL_L6, C_REFERENCE_ABS_SHEET_L6, C_REFERENCE_LLM_L6),
    ("L6 — order",        C_REFERENCE_FULL_L6, C_REFERENCE_ABS_SHEET_L6, C_REFERENCE_LLM_L6),
    ("L6 — genus",        C_REFERENCE_FULL_L6, C_REFERENCE_ABS_SHEET_L6, C_REFERENCE_LLM_L6),
    ("L6 — specie",       C_REFERENCE_FULL_L6, C_REFERENCE_ABS_SHEET_L6, C_REFERENCE_LLM_L6),
]:
    _ann = f"{C_ANNOTATED_PATH} [{_sh}]"
    _check_sources[_sheet] = {
        "Check 1: Manual vs Full-text": {"truth": _ft,  "estimate": _ann},
        "Check 2: LLM vs Full-text":    {"truth": _ft,  "estimate": _llm},
        "Check 3: Manual vs LLM":       {"truth": _ann, "estimate": _llm},
    }
# L3 region CC1 uses the archived geography file
# _check_sources["L3 — region"]["Check 1: Manual vs Full-text"]["truth"] = C_REFERENCE_FULL_L3_x

---
# L1 — Drivers

**Labels:** multi-label, each record may carry several driver categories.

**Special labels** (`not applicable`, `unclear`) appear only in manual annotations; they
are counted separately and excluded from per-label F1 but included in Jaccard.

## L1 — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [246]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH, 
                           sheet=C_REFERENCE_ABS_SHEET_L1, 
                           id_col="UT (Unique WOS ID)", 
                           label_col="driver"
                           )

    right = load_coding_df(C_REFERENCE_FULL_L1,
                           id_col="UT (Unique WOS ID)", 
                           label_col="driver")

    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L1 — Drivers"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L1 — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [247]:
left  = load_coding_df(C_REFERENCE_LLM_L1,  id_col="custom_id",          label_col="pred_driver")
right = load_coding_df(C_REFERENCE_FULL_L1, id_col="UT (Unique WOS ID)", label_col="driver")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L1 — Drivers"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 837


## L1 — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated (~69 of 100 annotated).

In [248]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L1,  id_col="custom_id",          label_col="pred_driver")
    right = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L1, id_col="UT (Unique WOS ID)", label_col="driver")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L1 — Drivers"].append(("Check 3: Manual vs LLM", summary, per_label))

---
# L2 — Threats

**Labels:** multi-label, each record may carry several threat categories (`threats_l1`).

**Special labels** (`not applicable`, `unclear`) appear only in manual annotations; they
are counted separately and excluded from per-label F1 but included in Jaccard.

## L2 / threats_l0 — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [249]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L2,
                           id_col="UT (Unique WOS ID)",
                           label_col="threats_l0")

    right = load_coding_df(C_REFERENCE_FULL_L2,
                           id_col="UT (Unique WOS ID)",
                           label_col="threats_l0")

    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L2 — threats_l0"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L2 / threats_l0 — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [250]:
# annotated_ids = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L2,
#                                id_col="UT (Unique WOS ID)", label_col="threats_l0")["_id"]

left  = load_coding_df(C_REFERENCE_LLM_L2,  id_col="custom_id",          label_col="pred_threat_l0")
# left  = left[left["_id"].isin(annotated_ids)]

right = load_coding_df(C_REFERENCE_FULL_L2, id_col="UT (Unique WOS ID)", label_col="threats_l0")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L2 — threats_l0"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 123


## L2 / threats_l0 — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [251]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L2,   id_col="custom_id",          label_col="pred_threat_l0")
    right = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L2, id_col="UT (Unique WOS ID)", label_col="threats_l0")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L2 — threats_l0"].append(("Check 3: Manual vs LLM", summary, per_label))

---
# L3 — Geography (Region, Sub-region, Country)

**Labels:** multi-label.  Three nested levels — `region`, `sub-region`, `country` — each
evaluated independently.

**Normalisation notes**
- Ground-truth region/sub-region strings are sometimes truncated (missing `]`) — repaired
  automatically by `_parse_multilabel`.
- LLM `pred_countries` are ISO-3166-alpha-3 codes (`"BHS"`, `"USA"`, …).  They are mapped
  to canonical country names via `ipbes_regions.json` before comparison.

**Special labels** (`not applicable`, `unclear`) and records where the ground-truth label
is empty after stripping are excluded from all metrics and counted separately.

In [252]:
# --- ISO-3166-alpha-3 → canonical country name (from IPBES regions mapping) ---
_cwd = Path.cwd().resolve()
_repo_root = _cwd if (_cwd / ".env").exists() else _cwd.parent
_env_path = _repo_root / ".env"
if _env_path.exists():
    for _raw_line in _env_path.read_text(encoding="utf-8").splitlines():
        _line = _raw_line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _key, _, _value = _line.partition("=")
        _key = _key.strip()
        _value = _value.strip()
        if _key:
            os.environ.setdefault(_key, _value)

_mappings_dir = os.environ["ORCHESTRATOR_MAPPINGS_DIR"].strip()
_mappings_dir = Path(_mappings_dir)
if not _mappings_dir.is_absolute():
    _mappings_dir = _repo_root / _mappings_dir

with open(_mappings_dir / "ipbes_regions.json") as _f:
    _ipbes = json.load(_f)

_iso_to_country = {}
for _subregions in _ipbes.values():
    for _countries in _subregions.values():
        for _c in _countries:
            if _c["ISO_3166_alpha_3"]:
                _iso_to_country[_c["ISO_3166_alpha_3"].lower()] = _c["Country"].lower()

def _norm_iso(df):
    """Replace ISO codes in _label frozensets with canonical country names."""
    df = df.copy()
    df["_label"] = df["_label"].apply(
        lambda s: frozenset(_iso_to_country.get(c, c) for c in s)
    )
    return df

## L3 / region — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [253]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L3,
                           id_col="UT (Unique WOS ID)", label_col="region")
    right = load_coding_df(C_REFERENCE_FULL_L3,  id_col="UT (Unique WOS ID)", label_col="region")
    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L3 — region"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L3 / region — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [254]:
left  = load_coding_df(C_REFERENCE_LLM_L3,  id_col="custom_id",          label_col="pred_regions")
right = load_coding_df(C_REFERENCE_FULL_L3, id_col="UT (Unique WOS ID)", label_col="region")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L3 — region"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 468


## L3 / region — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [255]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L3,   id_col="custom_id",          label_col="pred_regions")
    right = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L3, id_col="UT (Unique WOS ID)", label_col="region")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L3 — region"].append(("Check 3: Manual vs LLM", summary, per_label))

## L3 / sub-region — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [256]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L3,
                           id_col="UT (Unique WOS ID)", label_col="sub-region")
    right = load_coding_df(C_REFERENCE_FULL_L3,  id_col="UT (Unique WOS ID)", label_col="sub-region")
    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L3 — sub-region"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L3 / sub-region — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [257]:
left  = load_coding_df(C_REFERENCE_LLM_L3,  id_col="custom_id",          label_col="pred_subregions")
right = load_coding_df(C_REFERENCE_FULL_L3, id_col="UT (Unique WOS ID)", label_col="sub-region")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L3 — sub-region"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 468


## L3 / sub-region — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [258]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L3,   id_col="custom_id",          label_col="pred_subregions")
    right = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L3, id_col="UT (Unique WOS ID)", label_col="sub-region")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L3 — sub-region"].append(("Check 3: Manual vs LLM", summary, per_label))

## L3 / country — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`  
> Both sides use full country names — no ISO conversion needed.

In [259]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L3,
                           id_col="UT (Unique WOS ID)", label_col="country")
    right = load_coding_df(C_REFERENCE_FULL_L3,  id_col="UT (Unique WOS ID)", label_col="country")
    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L3 — country"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L3 / country — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`  
> LLM `pred_countries` are ISO codes — converted to full names via `_norm_iso`.

In [260]:
left  = _norm_iso(load_coding_df(C_REFERENCE_LLM_L3,  id_col="custom_id",          label_col="pred_countries"))
right =           load_coding_df(C_REFERENCE_FULL_L3, id_col="UT (Unique WOS ID)", label_col="country")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L3 — country"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 468


## L3 / country — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> LLM `pred_countries` are ISO codes — converted to full names via `_norm_iso`.  
> Note: only records present in both datasets are evaluated.

In [261]:
if MODE == "train":
    left  = _norm_iso(load_coding_df(C_REFERENCE_LLM_L3,   id_col="custom_id",          label_col="pred_countries"))
    right =           load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L3, id_col="UT (Unique WOS ID)", label_col="country")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L3 — country"].append(("Check 3: Manual vs LLM", summary, per_label))

---
# L4 — Ecosystems

**Labels:** multi-label. Two nested levels — `realm` and `biome` — each stored as a list of strings.

**Special labels** (`not applicable`, `unclear`) are stripped before metrics; records with an empty true label after stripping are excluded from evaluation.

## L4 / realm — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`

In [262]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L4,
                           id_col="UT (Unique WOS ID)",
                           label_col="realm")

    right = load_coding_df(C_REFERENCE_FULL_L4,
                           id_col="UT (Unique WOS ID)",
                           label_col="realm")

    m1 = merge_left(left, right)
    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L4 — realm"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L4 / realm — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm` (limited to annotated IDs)

In [263]:
left  = load_coding_df(C_REFERENCE_LLM_L4,  id_col="custom_id",          label_col="pred_realm")

right = load_coding_df(C_REFERENCE_FULL_L4, id_col="UT (Unique WOS ID)", label_col="realm")

m2 = merge_left(left, right)
summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L4 — realm"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 839


## L4 / realm — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`

In [264]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L4,   id_col="custom_id",          label_col="pred_realm")
    right = load_coding_df(C_ANNOTATED_PATH,  sheet=C_REFERENCE_ABS_SHEET_L4,
                           id_col="UT (Unique WOS ID)", label_col="realm")

    m3 = merge_left(left, right)
    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L4 — realm"].append(("Check 3: Manual vs LLM", summary, per_label))

---
# L5 — Study Design

**Labels:** multi-label (`study_design`). Each record may carry several study design categories.

**Special labels** (`not applicable`, `unclear`) are stripped before metrics; records with an empty true label after stripping are excluded from evaluation.

## L5 / study_design — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`

In [265]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L5,
                           id_col="UT (Unique WOS ID)",
                           label_col="study_design")

    right = load_coding_df(C_REFERENCE_FULL_L5,
                           id_col="UT (Unique WOS ID)",
                           label_col="study_design")

    m1 = merge_left(left, right)
    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L5 — study_design"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L5 / study_design — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm` (limited to annotated IDs)

In [266]:
# annotated_ids = load_coding_df(C_ANNOTATED_PATH, sheet=C_REFERENCE_ABS_SHEET_L5,
#                                id_col="UT (Unique WOS ID)", label_col="study_design")["_id"]

left  = load_coding_df(C_REFERENCE_LLM_L5,  id_col="custom_id",          label_col="pred_study_design")
# left  = left[left["_id"].isin(annotated_ids)]
right = load_coding_df(C_REFERENCE_FULL_L5, id_col="UT (Unique WOS ID)", label_col="study_design")

m2 = merge_left(left, right)
summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L5 — study_design"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 949


## L5 / study_design — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`

In [267]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L5,   id_col="custom_id",          label_col="pred_study_design")
    right = load_coding_df(C_ANNOTATED_PATH,  sheet=C_REFERENCE_ABS_SHEET_L5,
                           id_col="UT (Unique WOS ID)", label_col="study_design")

    m3 = merge_left(left, right)
    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L5 — study_design"].append(("Check 3: Manual vs LLM", summary, per_label))

---
# L6 — Taxa

**Labels:** multi-label, hierarchical. Six ranks — `kingdom`, `phylum`, `class`, `order`, `genus`, `specie` — evaluated independently (kingdom only in this initial pass).

**Special labels** (`not applicable`, `unclear`) appear only in manual annotations; they are stripped before metrics; records with an empty true label after stripping are excluded from evaluation.

## L6 / kingdom — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [268]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="kingdom")

    right = load_coding_df(C_REFERENCE_FULL_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="kingdom")

    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — taxa"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L6 / kingdom — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [269]:
left  = load_coding_df(C_REFERENCE_LLM_L6,  id_col="custom_id",          label_col="pred_kingdom")
right = load_coding_df(C_REFERENCE_FULL_L6, id_col="UT (Unique WOS ID)", label_col="kingdom")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L6 — taxa"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 146


## L6 / kingdom — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [270]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L6,   id_col="custom_id",          label_col="pred_kingdom")
    right = load_coding_df(C_ANNOTATED_PATH,  sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)", label_col="kingdom")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — taxa"].append(("Check 3: Manual vs LLM", summary, per_label))

## L6 / phylum — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [271]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="phylum")

    right = load_coding_df(C_REFERENCE_FULL_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="phylum")

    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — phylum"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L6 / phylum — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [272]:
left  = load_coding_df(C_REFERENCE_LLM_L6,  id_col="custom_id",          label_col="pred_phylum")
right = load_coding_df(C_REFERENCE_FULL_L6, id_col="UT (Unique WOS ID)", label_col="phylum")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L6 — phylum"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 146


## L6 / phylum — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [273]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L6,   id_col="custom_id",          label_col="pred_phylum")
    right = load_coding_df(C_ANNOTATED_PATH,  sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)", label_col="phylum")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — phylum"].append(("Check 3: Manual vs LLM", summary, per_label))

## L6 / class — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [274]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="class")

    right = load_coding_df(C_REFERENCE_FULL_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="class")

    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — class"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L6 / class — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [275]:
left  = load_coding_df(C_REFERENCE_LLM_L6,  id_col="custom_id",          label_col="pred_class")
right = load_coding_df(C_REFERENCE_FULL_L6, id_col="UT (Unique WOS ID)", label_col="class")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L6 — class"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 146


## L6 / class — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [276]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L6,   id_col="custom_id",          label_col="pred_class")
    right = load_coding_df(C_ANNOTATED_PATH,  sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)", label_col="class")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — class"].append(("Check 3: Manual vs LLM", summary, per_label))

## L6 / order — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [277]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="order")

    right = load_coding_df(C_REFERENCE_FULL_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="order")

    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — order"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L6 / order — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [278]:
left  = load_coding_df(C_REFERENCE_LLM_L6,  id_col="custom_id",          label_col="pred_order")
right = load_coding_df(C_REFERENCE_FULL_L6, id_col="UT (Unique WOS ID)", label_col="order")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L6 — order"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 146


## L6 / order — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [279]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L6,   id_col="custom_id",          label_col="pred_order")
    right = load_coding_df(C_ANNOTATED_PATH,  sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)", label_col="order")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — order"].append(("Check 3: Manual vs LLM", summary, per_label))

## L6 / genus — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [280]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="genus")

    right = load_coding_df(C_REFERENCE_FULL_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="genus")

    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — genus"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L6 / genus — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [281]:
left  = load_coding_df(C_REFERENCE_LLM_L6,  id_col="custom_id",          label_col="pred_genus")
right = load_coding_df(C_REFERENCE_FULL_L6, id_col="UT (Unique WOS ID)", label_col="genus")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L6 — genus"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 146


## L6 / genus — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [282]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L6,   id_col="custom_id",          label_col="pred_genus")
    right = load_coding_df(C_ANNOTATED_PATH,  sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)", label_col="genus")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — genus"].append(("Check 3: Manual vs LLM", summary, per_label))

## L6 / specie — Consistency Check #1
**Manual abstract annotations (`c_reference_abs`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_abs`  
Join: `c_reference_abs` LEFT JOIN `c_reference_full`

In [283]:
if MODE == "train":
    left  = load_coding_df(C_ANNOTATED_PATH,
                           sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="specie")

    right = load_coding_df(C_REFERENCE_FULL_L6,
                           id_col="UT (Unique WOS ID)",
                           label_col="specie")

    m1 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m1["y_true"], m1["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — specie"].append(("Check 1: Manual vs Full-text", summary, per_label))

## L6 / specie — Consistency Check #2
**LLM coding (`c_reference_llm`) vs full-text train labels (`c_reference_full`)**

Truth: `c_reference_full` &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_full`

In [284]:
left  = load_coding_df(C_REFERENCE_LLM_L6,  id_col="custom_id",          label_col="pred_species")
right = load_coding_df(C_REFERENCE_FULL_L6, id_col="UT (Unique WOS ID)", label_col="specie")
m2 = merge_left(left, right)

summary, per_label = multilabel_metrics(m2["y_true"], m2["y_pred"])
# display(summary)
# display(per_label)
_results["L6 — specie"].append(("Check 2: LLM vs Full-text", summary, per_label))

  Matched: 146


## L6 / specie — Consistency Check #3
**Manual abstract annotations (`c_reference_abs`) vs LLM coding (`c_reference_llm`)**

Truth: `c_reference_abs` (manual) &nbsp;|&nbsp; Estimate: `c_reference_llm`  
Join: `c_reference_llm` LEFT JOIN `c_reference_abs`  
> Note: only records present in both datasets are evaluated.

In [285]:
if MODE == "train":
    left  = load_coding_df(C_REFERENCE_LLM_L6,   id_col="custom_id",          label_col="pred_species")
    right = load_coding_df(C_ANNOTATED_PATH,  sheet=C_REFERENCE_ABS_SHEET_L6,
                           id_col="UT (Unique WOS ID)", label_col="specie")
    m3 = merge_left(left, right)

    summary, per_label = multilabel_metrics(m3["y_true"], m3["y_pred"])
    # display(summary)
    # display(per_label)
    _results["L6 — specie"].append(("Check 3: Manual vs LLM", summary, per_label))

# Output

In [286]:
# --- Save results to xlsx ---
OUTPUT_DIR  = f"../data/labels/eval/consistency-check/coding/{RUN_NAME}"
OUTPUT_XLSX = f"{OUTPUT_DIR}/coding_results.xlsx"

os.makedirs(OUTPUT_DIR, exist_ok=True)
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for sheet_name, checks in _results.items():
        row = 0
        for check_name, summary_df, per_label_df in checks:
            pd.DataFrame([[f"── {check_name} ──"]]).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False, header=False)
            row += 1
            src = _check_sources.get(sheet_name, {}).get(check_name, {})
            if src:
                pd.DataFrame([
                    [f"Truth:    {src['truth']}"],
                    [f"Estimate: {src['estimate']}"],
                ]).to_excel(writer, sheet_name=sheet_name, startrow=row, index=False, header=False)
                row += 2
            summary_df.to_excel(writer, sheet_name=sheet_name, startrow=row, index=False)
            row += len(summary_df) + 2
            pd.DataFrame([["Per Label"]]).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False, header=False)
            row += 1
            per_label_df.to_excel(writer, sheet_name=sheet_name, startrow=row, index=False)
            row += len(per_label_df) + 3

print(f"Save - {OUTPUT_XLSX}")

Save - ../data/labels/eval/consistency-check/coding/180426_gpt5nanoHigh_test/coding_results.xlsx
